# 02.1 The ndarray Memory Model: Buffer, Dtype, Strides — Views vs Copies

> **Prerequisites:** 01.2 (the data universe; the ingestion boundary and the reconciliation
> guard) · 01.3 (reproducibility; content-hashing what a run consumed)
> **What you'll learn:**
> - Predict, before running it, whether a numpy operation returns a view or a copy — and settle it with `np.shares_memory` instead of folklore
> - Read `(offset, shape, strides)` well enough to explain why transposing 286,432 rows is free and selecting 3 of them is not
> - Catch mutation-at-a-distance: the in-place write through a view that corrupts an array nobody touched
> - Port 2019-era array code across numpy 2.x's `copy=` and dtype semantics without silent behaviour changes
> - Serve canonical arrays read-only, with mutation pushed behind one explicit copy at the boundary
> **Level:** Beginner · **Series:** 02 NumPy & Vectorized Computing

> ⚡ **Monday 2026-08-31, 07:55** — the weekly USD collections report runs, and it is correct.
> At 08:10 the reconciliation job, reading the *same in-memory payments array*, reports a
> total 41 million below the 07:40 snapshot of that same array. The export files on disk
> hash identical to yesterday's. The cause: a code review removed a "redundant" `.copy()`
> two weeks ago, and one job's scratch buffer turned out to be another job's source of truth.


## Concept
### Plain-English Explanation

Series 01 treated arrays as things that hold numbers. This notebook is about what an array
actually *is*, because the difference decided the cold open. A numpy array is two separable
things: a flat run of bytes (the **buffer**) and a small header that says how to read it — a
starting position, a shape, a step size per axis, and a rule for turning bytes into values.
Many arrays can share one buffer while carrying different headers. That sharing is what makes
numpy fast, and it is also a standing invitation to mutate data at a distance: write through
any one of those headers and every other one sees the change, because there was only ever one
set of bytes.

The engineering question of this notebook is therefore not "how do I slice an array" — the
2019 notes covered that — but "when I slice, who else is holding these bytes, and what happens
when one of us writes?" Getting that wrong does not crash. It produces a second, silently
different answer from data nobody believes they touched, which is the most expensive kind of
bug series 01 met: the kind where every individual value stays plausible.

### Technical Explanation

An ndarray is a view descriptor over a buffer: **`offset`** (where element `[0, 0, ...]`
lives), **`shape`** (how many elements per axis), **`strides`** (how many *bytes* to step to
move one element along each axis), and **`dtype`** (how many bytes make one value, and how to
interpret them). Every indexing expression numpy evaluates reduces to one formula: the address
of element `(i, j)` is `offset + i*strides[0] + j*strides[1]`. Nothing else exists — no nested
lists, no per-row objects, just that arithmetic over one allocation.

Two consequences carry the whole notebook. First, any operation whose result is expressible as
*new numbers in the header over the same buffer* can be free: transposing swaps the two stride
values, slicing moves the offset and shrinks the shape, and neither touches a byte of data.
numpy calls the result a **view**, and marks it by setting its `.base` to the owning array.
Second, any operation whose result *cannot* be described by constant strides — selecting rows
`[2, 0, 7]`, keeping only the elements above a threshold — must allocate a fresh buffer and
copy into it. The first kind aliases; the second is independent. ⭐ **CRITICAL CONCEPT** — the
view/copy distinction is not an API detail to memorise per function; it is a single rule about
whether the selection has a constant stride, and `np.shares_memory(a, b)` answers it
mechanically whenever you are unsure.

The numbers that matter here: the PayFlow payments export is 286,432 rows; as a float64 array
its buffer is 2.3 MB at 8 bytes per element. A 3×4 float64 block carries strides `(32, 8)` —
one row is four 8-byte elements away, one column is one element away — and its transpose
carries `(8, 32)`: the same two numbers, swapped, over the same 96 bytes. A view of the full
array costs the same three header updates whether the array holds a dozen rows or a hundred
million; a copy moves the buffer, and its cost grows with `n`.

### Mental Model

An ndarray is a *reading instruction* pointed at a buffer, and most "operations" just write a
new instruction. Ask of every array you are about to write into: whose bytes are these? If you
cannot answer, `np.shares_memory` can — and the write should probably wait until you have.


## How It Works

```text
  the buffer: one contiguous run of bytes (payments amount_paid, float64, 8 bytes each)

  [ 496.31 | 494.83 | 546.47 | 560.45 | 542.54 | 545.59 | ... 286,432 values ... ]

  an ARRAY is a header over it:   (offset, shape, strides, dtype)

    m = a[:12].reshape(3, 4)      offset 0     shape (3,4)   strides (32, 8) bytes
        element (i, j) lives at   offset + i*32 + j*8    <- ALL indexing is this formula

    m.T                           offset 0     shape (4,3)   strides (8, 32)
    m[1:3]                        offset 32    shape (2,4)   strides (32, 8)
        ^ header-only edits: ZERO bytes move, result ALIASES the buffer (a view)

    m[[2, 0]]   m[mask]           no constant stride can describe the selection
        ^ fresh buffer, elements copied in: independent of the source (a copy)

  writes through ANY view land in THE buffer - every other header sees them
```

Read the formula line first, because everything else is a corollary. Moving one step down a
row of `m` must skip a full row of the buffer — four elements, 32 bytes — while one step along
the row skips one element, 8 bytes. That is all `strides` says. Transposition swaps the roles
of the two axes, so it swaps the two numbers; slicing rows starts reading 32 bytes later and
claims fewer of them. Neither operation looks at a single value, which is why both cost the
same on twelve elements as on a hundred million.

The copy cases are forced, not chosen. `m[[2, 0]]` asks for row 2 and then row 0 — a step of
*minus* two rows — and `m[mask]` keeps an irregular subset; no `(offset, shape, strides)`
triple describes either walk, so numpy allocates and copies. This is also why the two kinds of
indexing behave differently as *assignment targets*, a trap 02.4 measures in full.

The cold open follows directly. The weekly report took the export's newest 1,200 rows as
`day = amounts[-1200:]` — a header-only edit, aliasing the canonical buffer — and then ran
`day /= fx`, an **in-place** operation that writes each result back through the header into
the shared bytes. The report printed correct USD numbers and quietly converted 1,200 elements
of everyone else's local-currency array. The 08:10 job read the same buffer and got a
different total. No file changed; no exception was raised; every mutated value remained a
plausible payment.


## Hands-On Build
### Stage A — from scratch

Strides feel like trivia until you implement them once. The class below is a 2-D "array" over
a flat Python list: indexing, transposition and row-slicing as pure bookkeeping over
`(offset, shape, strides)`, with strides in elements rather than bytes so the arithmetic stays
legible. If its answers match numpy's on the same values, the mechanism is understood.


In [1]:
# Load the committed lab module; it owns the stdlib CSV parsing (M7 commas stripped
# at the boundary - 01.2's contract lesson) and every experiment this notebook shows.
import importlib.util
import sys
from pathlib import Path

import numpy as np

LAB = Path.cwd() / "_lab" / "lab_02.1_ndarray_memory.py"
spec = importlib.util.spec_from_file_location("lab_02_1", LAB)
lab = importlib.util.module_from_spec(spec)
sys.modules["lab_02_1"] = lab
spec.loader.exec_module(lab)

amounts, fx, commas = lab.load_payments()
print(f"payments rows={len(amounts):,}  dtype={amounts.dtype}  "
      f"itemsize={amounts.itemsize} bytes  buffer={amounts.nbytes / 1e6:,.1f} MB")
print(f"comma-formatted amounts parsed at the boundary (M7): {commas:,}")

payments rows=286,432  dtype=float64  itemsize=8 bytes  buffer=2.3 MB
comma-formatted amounts parsed at the boundary (M7): 25,351


Two of those numbers do quiet work later. The 25,351 comma-formatted amounts are M7 —
the same defect that turned 01.2's revenue sum into a string — handled here in the loader,
once, at the boundary. And 2.3 MB is worth holding onto: every "is this a copy?" question in
this notebook is ultimately a question about whether those bytes move.

Now the Stage A build, on the first twelve of those values.


In [2]:
class StridedView:
    """A 2-D 'array' over a flat buffer: nothing but (offset, shape, strides).

    Strides here are in ELEMENTS (numpy's are in bytes: multiply by itemsize).
    Every operation edits these three numbers; the buffer is never touched.
    """

    def __init__(self, buf, offset, shape, strides):
        self.buf, self.offset, self.shape, self.strides = buf, offset, shape, strides

    def __getitem__(self, ij):
        i, j = ij
        if not (0 <= i < self.shape[0] and 0 <= j < self.shape[1]):
            raise IndexError(f"({i},{j}) outside {self.shape}")
        return self.buf[self.offset + i * self.strides[0] + j * self.strides[1]]

    def transpose(self):
        """Swap shape and strides. No element moves."""
        return StridedView(self.buf, self.offset,
                           (self.shape[1], self.shape[0]),
                           (self.strides[1], self.strides[0]))

    def rows(self, a, b):
        """Row-slice: move the offset, shrink the shape. No element moves."""
        return StridedView(self.buf, self.offset + a * self.strides[0],
                           (b - a, self.shape[1]), self.strides)

In [3]:
flat = amounts[:12].tolist()                       # one shared flat buffer
mine = StridedView(flat, offset=0, shape=(3, 4), strides=(4, 1))
ref = amounts[:12].reshape(3, 4)                   # numpy over the same values

for name, got, want in [
        ("element (1,2)", mine[1, 2], float(ref[1, 2])),
        ("transpose (2,1)", mine.transpose()[2, 1], float(ref.T[2, 1])),
        ("rows(1,3) at (0,3)", mine.rows(1, 3)[0, 3], float(ref[1:3][0, 3]))]:
    assert got == want, (name, got, want)
    print(f"{name:<20} hand-strided={got:>8.2f}   numpy={want:>8.2f}   MATCH")

print(f"\nnumpy strides for the same 3x4 float64 block: {ref.strides} bytes"
      f"  (= elements {tuple(s // ref.itemsize for s in ref.strides)} x itemsize 8)")
print(f"transpose strides: {ref.T.strides} bytes - the same two numbers, swapped")

element (1,2)        hand-strided=  560.70   numpy=  560.70   MATCH
transpose (2,1)      hand-strided=  560.70   numpy=  560.70   MATCH
rows(1,3) at (0,3)   hand-strided=  504.41   numpy=  504.41   MATCH

numpy strides for the same 3x4 float64 block: (32, 8) bytes  (= elements (4, 1) x itemsize 8)
transpose strides: (8, 32) bytes - the same two numbers, swapped


The three MATCH lines are the proof: hand-computed `offset + i*stride` arithmetic
reproduces numpy exactly, including after a transpose and a row-slice, because that arithmetic
is all numpy does for these operations. And the strides printout confirms the diagram —
`(32, 8)` becomes `(8, 32)` under `.T`, the same two integers swapped over the same buffer,
which is why transposing never gets slower as arrays grow. (The committed lab runs this same
parity check standalone, so the claim survives outside this notebook.)

### Stage B — idiomatic

With the mechanism proven, the practical question is coverage: *which* numpy operations alias
and which allocate. Folklore answers this badly — "slices are views, everything else copies"
is wrong in both directions — so the census below settles each case with `np.shares_memory`,
the same tool you should reach for in a debugger.


In [4]:
lab.view_copy_census(amounts)

  operation                    shares_memory  .base set   verdict
  basic slice a[1000:2000]              True       True   VIEW - writes reach the source
  strided slice a[::50]                 True       True   VIEW - writes reach the source
  reshape (on contiguous)               True       True   VIEW - writes reach the source
  transpose m.T                         True       True   VIEW - writes reach the source
  ravel of contiguous m                 True       True   VIEW - writes reach the source
  ravel of m.T (non-contig)            False      False   copy - independent
  boolean mask a[a > 1000]             False      False   copy - independent
  fancy index a[[1, 5, 7]]             False      False   copy - independent
  explicit a.copy()                    False      False   copy - independent

  the rule underneath: an operation returns a view exactly when the result
  is expressible as (new offset, new shape, new strides) over the SAME buffer.
  Basic slicing and transp

⚠️ Read the two `ravel` rows as a pair, because they break the folklore cleanly. On the
contiguous block, `ravel` is a header edit and aliases; on the transpose of that same block it
must interleave elements no constant stride can describe, so *the same function call* silently
allocates. Whether you got a view depends on the layout of the argument, not on the function's
name — which is exactly why the census's verdict column comes from `shares_memory` and not
from documentation memory. The `.base` column is the cheap first probe (`None` means the array
owns its buffer), and `np.shares_memory` is the decisive one.

The rule stated under the table is the one worth keeping: **view when the selection is
expressible as constant strides over the same buffer, copy when it is not.** Basic slices and
transposes always are. Boolean and fancy selection never are. `reshape` and `ravel` are
whenever layout permits.

Views exist because that header-edit trick is enormously cheaper than moving bytes — the cell
below prices it, and the price explains why numpy chose aliasing-by-default even though this
whole notebook is about its cost.


In [5]:
lab.view_vs_copy_timing(amounts)

  slice a[:286,431] as a view :      0.50 us   (three integers change)
  the same slice, copied        :    579.90 us   (2.3 MB moves)
  ratio ~1,160x on this run - machine- and run-dependent, an order of magnitude at least; the view's cost does not
  grow with n, the copy's does. Aliasing is a PERFORMANCE feature with
  CORRECTNESS obligations - this notebook is about paying them.


The exact microseconds are machine- and run-dependent (01.4's benchmark honesty applies
— rerun the cell and the numbers move), but the shape of the comparison is not: the view costs
the same three header updates at any array size, while the copy moves the full 2.3 MB and
scales with `n`. Aliasing is a performance feature with correctness obligations. The 2019
notes taught the feature; the obligations are the rest of this notebook.

One more Stage B block, because two of those obligations changed between the 2019 notes and
the pinned numpy 2.5 — the migration thread this series carries (P5).


In [6]:
lab.migration_semantics(amounts)

  np.array(copy=False) with a cast RAISES in 2.x - 'copy if you must'
    is now spelled copy=None; the 2019 notes' idiom is a hard error
  b.dtype = np.int64 -> DeprecationWarning (numpy 2.5): mutating a possibly-shared
    array's interpretation is unsafe; the supported spelling is a NEW view:
    a[:2]              = [496.31 494.83]  (float64 values)
    a.view(np.int64)[:2] = [4647438794047314985 4647412757611969249]  (the SAME bytes read as int64)
    shares_memory=True, itemsize 8->8: reinterpretation, not conversion - .astype() converts


Three behaviours, three eras. **`copy=False` hardened.** In 1.x it meant "avoid a copy
if you can", so code written against the 2019 notes could receive a copy anyway and mutate it
in false safety; in 2.x it is a promise numpy refuses to break, raising when a cast forces a
copy — "copy if you must" is now spelled `copy=None`. **Assigning to `.dtype` is deprecated in
numpy 2.5** for precisely this notebook's reason: it mutates the *interpretation* of a buffer
that other views may share. **The supported spelling, `.view(dtype)`, reinterprets bytes
rather than converting values** — the cell shows the first two payments, `496.31` and
`494.83`, reading back as 19-digit integers, because those are the int64 readings of the same
eight bytes. When you want the *values* as another type, `.astype()` converts and always
copies; `.view(dtype)` is for when you genuinely mean the bytes.

### Stage C — production

Everything above says the same thing: shared bytes plus an in-place write is a loaded gun. The
production answer is not "be careful", it is a contract — canonical arrays are served
**read-only**, and mutation happens only on the far side of one explicit, named copy.


In [7]:
lab.canonical_contract()

  the incident's exact code, against the read-only canonical array:
    ValueError: output array is read-only
    the failure moved from a reconciliation gap at 08:10 to a stack
    trace at the offending line - detection distance zero
  the sanctioned path: working_copy() then convert -> USD total 2,349,191.59, canonical untouched
    [PASS] canonical is read-only
    [PASS] views inherit read-only
    [PASS] working_copy is independent
    [PASS] working_copy is writable


The first block is the payoff: the cold open's exact code — a tail slice, an in-place
divide — now dies with `ValueError: output array is read-only` at the offending line, instead
of surfacing as a reconciliation gap two reads later. The flag costs nothing (it is
bookkeeping in the header, not a copy), views inherit it, and `working_copy()` is the single
sanctioned door back to mutability: an explicit `.copy()` whose name says what it is for.

## Evaluation

A toolkit notebook's mechanism is exercised by assertions rather than by a model metric
(guide §2: for series 01–08 the eval may be unit-test-shaped, and here it is). The harness is
three layers, all captured above: the Stage A parity assertions, which fail the build if the
hand-strided arithmetic ever disagrees with numpy; the census, whose every row is a
`shares_memory` check that would catch a behaviour change across numpy versions (rerun it on
any upgrade — it is nine lines and it is exactly what the 2.x migration changed); and the four
`[PASS]` contract tests, which are the production guard expressed as one-line assertions a CI
job runs on every build. A meaningful "delta" here is any of those lines flipping — there is
no noise band to clear, because nothing here is stochastic.


## Design Patterns / Tradeoffs

**Views-by-default versus copy-on-write.** numpy aliases wherever strides allow and asks you
to track ownership; the win is that slicing never costs memory or time, which is why a 2.3 MB
array can be windowed thousands of times for free, and the loss is this notebook's incident
class. The opposite design — every derived array copies when first written — buys safety and
pays in memory and in copies you did not ask for. pandas 3 made that opposite choice
(copy-on-write is its default, taught properly in 03.1), which means the two libraries your
pipeline uses daily now sit on *opposite* sides of this trade: a numpy slice aliases, a pandas
selection behaves like an independent value. Knowing which world each object lives in is now
part of reading code. Use numpy's aliasing deliberately — for windows, for zero-cost
reshaping, for large read paths; treat it as hostile anywhere a mutable array crosses a
function boundary.

**Read-only canonical arrays versus defensive copying.** Defensive copying — every function
copies its inputs "to be safe" — is the common reflex, and at PayFlow's scale it is the wrong
one: it hides the ownership question instead of answering it, and it turns a 2.3 MB buffer
into dozens of them. The contract above inverts the default: the canonical array is immutable
(one flag), aliasing is safe *because* writes are impossible, and the code that genuinely
needs mutation declares it by calling `working_copy()`. The cost is honesty about a boundary —
someone must decide which arrays are canonical. Prefer read-only-plus-explicit-copy for shared
reference data; accept defensive copies only where arrays are small and ownership genuinely
cannot be pinned down.

**`.view(dtype)` versus `.astype(dtype)`.** Reinterpretation versus conversion. `.astype`
answers "these values, in that type" and always copies; `.view` answers "these bytes, read
differently" and never does. The 2019-era idiom of assigning to `.dtype` sat ambiguously
between them and is now deprecated (numpy 2.5) for exactly that ambiguity. Reach for `.view`
only when the bytes themselves are the object — binary formats, hashing (01.3's digests do
this), micro-optimizing a reinterpret — and let everything else be `.astype`.

**Recommendation for PayFlow:** canonical exports load once, read-only, behind the Stage C
loader; every consumer that mutates goes through `working_copy()`; `shares_memory` assertions
ride in the test suite next to 01.2's reconciliation; and any in-place operator (`/=`, `*=`,
`[...] =`) on an array received from elsewhere is a code-review flag until someone names whose
bytes it writes.


## Production Scenario
### Symptoms

**Monday 2026-08-31, 07:55.** The weekly USD collections report and the reconciliation job
share one process and one canonical payments array, loaded from the append-only export at
07:30. A fortnight ago, a tidy-up PR removed a `.copy()` on the report's input slice with the
comment "avoid a redundant copy of 1,200 rows".

- **07:55** — the weekly report completes. Its USD figures are correct to the cent; nobody
  will ever file a ticket about the report.
- **08:10** — the reconciliation job, reading the same in-memory array, reports a collected
  total 41,364,097.82 below the 07:40 integrity snapshot of the *same array*. (Both totals
  are mixed-currency sums — meaningless as money, per 01.2's category error, but perfectly
  serviceable as a checksum: any stable function of the buffer works as a tripwire.)
- The export files on disk hash identical to Friday's manifest (01.3's `frame_hash`
  discipline): whatever changed, it did not come through the filesystem.
- Job logs are clean end to end. No exception, no warning, no retry. Every payment value in
  the array remains individually plausible — nothing is zeroed, negated or NaN.
- The gap is stable on re-read at 08:25 and does not grow: whatever happened, happened once.


In [8]:
summary = lab.incident(amounts, fx)

  canonical local-currency total, 07:40 snapshot :    15,014,288,619.50
  daily USD report built from the tail increment  :         2,349,191.59  (the report itself is right)
  canonical local-currency total, 08:10 read      :    14,972,924,521.68
  the two reads of the SAME array differ by       :        41,364,097.82
  np.shares_memory(canonical, day) = True <- the smoking gun
  nothing on disk changed; the mutation happened in memory, through a view,
  on 1,200 of 286,432 rows - every value still plausible


### Diagnosis

Walking the ladder in its data-pipeline form, naming what each signal eliminated:

1. **Alert** — the reconciliation delta between two reads of one array. Candidate causes: the
   input files changed between loads, a bug wrote garbage, a job read a different array, or
   something mutated the shared buffer in memory.
2. **Job logs** — clean. This eliminates crash-and-partial-write and anything that raises;
   whatever wrote, wrote through an API that considers the operation legitimate.
3. **Input-data checks** — the on-disk exports hash identical to the recorded manifest, so
   the mutation did not arrive through a reload. The array changed *in memory*, between two
   timestamps, in a process that "only read" it. This is the pivotal elimination: it converts
   a data-quality investigation into a memory-ownership one.
4. **Value inspection** — diff the 07:40 snapshot against the 08:10 state: exactly 1,200
   trailing rows differ, and each new value equals the old divided by that row's `fx_rate_usd`.
   The change is not corruption; it is a *transformation* — specifically the USD conversion
   the weekly report performs. The blast radius is exactly the report's input window.
5. **Code diff** — the fortnight-old tidy-up PR: `day = amounts[-1200:].copy()` became
   `day = amounts[-1200:]`, and the conversion below it is in-place (`day /= fx_tail`).
   `np.shares_memory(canonical, day)` returns `True` — the smoking gun the cell above prints.
6. **Mechanism confirmed** — basic slicing returns a view; `/=` writes each result back
   through the view's header into the shared buffer. The report was correct *and* the source
   was corrupted, by the same statement.

### Root Cause

`amounts[-1200:]` is a header-only view over the canonical buffer, and the in-place divide
wrote USD values through it into 1,200 shared elements. The removed `.copy()` had been the
only thing separating the report's scratch space from the source of truth; removing it changed
no output of the report — which is why the PR looked safe and its tests passed — while turning
the report into an unrequested writer of everyone else's data.

### Fix

**Mitigation now.** Reload the canonical array from the export (disk is intact — step 3 proved
it) and re-run the 08:10 job; quarantine the report's in-place conversion behind an explicit
copy restored at the call site.

**Permanent fix.** The Stage C contract: the loader serves canonical arrays with
`writeable=False`, so the incident's exact code now raises `ValueError: output array is
read-only` at the offending line; mutation moves behind `working_copy()`, a named, explicit
copy at the boundary. The four contract assertions join the test suite.

### Prevention

- **Read-only by default for shared reference arrays.** The flag is free, views inherit it,
  and it converts this incident class from a reconciliation mystery into a stack trace —
  detection distance zero, in 01.6's vocabulary.
- **`shares_memory` assertions in tests** wherever an array crosses an ownership boundary:
  one line, and it fails the exact PR that caused this.
- **The integrity snapshot stays.** Two reads of a canonical array disagreeing is what caught
  this at 08:10 rather than at month-end; 01.2's reconciliation is the same idea one level up.
- **Review rule:** an in-place operator on an array the function did not allocate is a
  question, not a style choice — "whose bytes are these?" must have an answer in the PR.


## Common Pitfalls

⚠️ **Believing a slice is a copy.** The single most common numpy error in 2019-era code and
the cold open's entire mechanism. Basic slices alias; if you are about to write, take
`working_copy()` or prove ownership with `np.shares_memory`.

⚠️ **"I checked, and this function returns a view" — on your layout.** `ravel` on a contiguous
array aliases and on its transpose copies; `reshape` likewise. View-ness is a property of the
argument's layout, not of the function name, so the check that matters is on *your* array, in
*this* code path.

**Testing aliasing with `is` or `==`.** Two different array objects can share one buffer, so
identity comparison answers the wrong question and `==` compares values elementwise.
`np.shares_memory(a, b)` (or the cheap first probe, `b.base is not None`) answers the real one.

**Mutating through a "temporary" in a chained expression.** In-place operators on anything
derived from a shared array write through to it. The dangerous shape is exactly the incident's:
`tail = big[-n:]; tail /= rate` — two innocent lines whose combination is a write to `big`.

⚠️ **Porting 1.x `copy=False` forward untouched.** It silently *may* have copied in 1.x; in
2.x it raises the moment a cast makes the no-copy promise unkeepable. The 2019 notes' idiom
must become `copy=None` ("avoid if possible") or an explicit `.copy()` — the migration is
mechanical, but only once you know which meaning each call site wanted.

**Assigning to `.dtype` to reinterpret.** Deprecated in the pinned numpy 2.5 because it
mutates shared state; and half the time the author wanted `.astype` (convert values) anyway,
not reinterpretation. Decide which you mean, then say it: `.astype(t)` or `.view(t)`.

**Copying everything defensively "to be safe".** The opposite failure: correctness by
memory explosion, and the ownership question is still unanswered — just hidden. Prefer one
read-only canonical array and explicit copies at named boundaries.


## Interview Questions

1. **Derive this.** For a C-ordered array of shape `(r, c)` and itemsize `s`, derive the
   strides, the address of element `(i, j)`, and the strides after a transpose — then explain
   why the transpose costs O(1). *Answer shape:* row stride `c*s`, column stride `s`; address
   `offset + i*c*s + j*s`; transpose swaps the pair to `(s, c*s)` over the same buffer, so
   only the header changes regardless of `r*c`.
2. **Design this.** Several jobs in one process share a large reference array; some need
   mutable scratch derived from it. Design the ownership contract. *Answer shape:* serve the
   canonical array read-only (`writeable=False`, inherited by views); one named
   `working_copy()` boundary for mutation; `shares_memory` assertions at hand-off points in
   tests; in-place ops on non-owned arrays treated as review flags.
3. **Debug this.** Two reads of the same in-memory array, minutes apart, give different
   totals; files on disk are unchanged and logs are clean. Walk your diagnosis. *Answer
   shape:* hash the on-disk inputs against the manifest to rule out a reload; diff the two
   states to localize and characterise the change (which rows, transformed how); search for
   views of the buffer (`shares_memory`) and recent diffs that removed copies or introduced
   in-place ops; confirm by reproducing the transformation through the suspect view.
4. When does `reshape` return a view, and when must it copy? *Answer shape:* a view exactly
   when the requested shape is reachable with constant strides over the existing layout —
   contiguous cases yes; after transposes or slicings that break constant stride, it copies.
   Verify with `shares_memory`, not by rule of thumb.
5. What is the difference between `a.view(np.int64)` and `a.astype(np.int64)` on a float64
   array? *Answer shape:* `view` reinterprets the same bytes under a new dtype — free,
   aliasing, and numerically meaningless unless you wanted the bytes; `astype` converts each
   value into a new buffer. The deprecated `.dtype =` assignment was in-place
   reinterpretation, unsafe on shared buffers.
6. Why did numpy 2.x make `np.array(x, copy=False)` raise where 1.x sometimes copied
   anyway? *Answer shape:* the 1.x behaviour made "I hold a view" unverifiable — code could
   receive a silent copy and mutate it believing writes reached the source, or vice versa;
   2.x turns the flag into a checkable promise and moves the old best-effort meaning to
   `copy=None`.
7. Your colleague "optimises" a pipeline by removing `.copy()` calls, and every test still
   passes. What risk did the tests miss, and what test catches it? *Answer shape:* outputs
   are unchanged — the removed copies changed *aliasing*, not values, so downstream arrays
   are now silently shared; a `shares_memory` assertion at each boundary (or serving sources
   read-only) fails the exact commit.


## Key Takeaways

- An ndarray is `(offset, shape, strides, dtype)` over a shared buffer; every indexing
  operation is one multiply-add per axis, and most "operations" only edit the header.
- View when the selection is expressible as constant strides over the same buffer, copy when
  it is not — basic slices and transposes alias, boolean and fancy selection never do, and
  `reshape`/`ravel` depend on layout, so settle real cases with `np.shares_memory`.
- Writes through any view land in the shared buffer: an in-place operator on a slice is a
  write to the source, which is exactly how a correct report corrupted its own input.
- Views are O(1) at any size and copies scale with the buffer — aliasing is a performance
  feature carrying correctness obligations, so use it on read paths and gate it on write paths.
- Serve shared reference arrays read-only and route mutation through one explicit
  `working_copy()` boundary; the flag is free and turns this incident class into a stack trace.
- Migrating 2019 code: `copy=False` now raises rather than maybe-copying (`copy=None` is the
  old meaning), assigning `.dtype` is deprecated in 2.5, and the replacement question is
  always "bytes (`.view`) or values (`.astype`)?"
- When memory questions arise, measure rather than recall: `.base`, `np.shares_memory`,
  `.flags`, `.strides` answer in one line what folklore gets wrong in both directions.


## Related

**Backward**

- **01.2 First Contact with the PayFlow Data Universe** — the ingestion-boundary discipline
  (M7 commas parsed once, at the edge) and the reconciliation habit that caught this
  incident; its `Money` lesson is why the checksum totals here are labelled non-money.
- **01.3 Reproducibility as an Engineering Contract** — the on-disk content hashes that let
  Diagnosis step 3 rule out the filesystem in one line.

**Forward**

- **02.2 Numerical Dtypes & Promotion under NEP 50** — dtype as *interpretation* becomes
  dtype as *arithmetic behaviour*: promotion, overflow and the Windows int64 change.
- **02.4 Indexing & Selection** — the assignment side of today's boundary: why writes into
  fancy-indexed selections vanish, and the masks that replace them.
- **02.7 Memory Layout & Performance** — strides meet caches: contiguity, temporaries, and
  when the free transpose stops being free.
- **03.1 Pandas & Modern DataFrames** — the opposite design decision: pandas 3's
  copy-on-write default, and reading code that mixes both memory models.
